# Progree AI Internship - Task 2
## Intelligent Multi-Class Text Sentiment Classifier

**Pipeline:** clean text → remove stop-words → lemmatize → TF-IDF → train 3 models → evaluate with F1-score → save results.

**How to use in Colab:** run the cells from top to bottom (Runtime → Run all). In Step 2 you can upload your own CSV, or just use the built-in demo data to test that everything works.

## Step 1: Setup

In [ ]:
import os, re, json, random, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)

for pkg in ["stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

RANDOM_STATE = 42
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
print("Setup done ✅")

## Step 2: Load the dataset

Set `USE_DEMO = False` and run the cell to upload your CSV (for example Kaggle's **Twitter US Airline Sentiment** `Tweets.csv`).
Then set `TEXT_COL` and `LABEL_COL` to your column names (for `Tweets.csv`: `text` and `airline_sentiment`).

Leave `USE_DEMO = True` for a quick test. Demo scores will be unrealistically perfect, so use a real dataset for submission.

In [ ]:
USE_DEMO = True          # <-- change to False to use your own CSV
TEXT_COL = "text"
LABEL_COL = "airline_sentiment"

def build_demo_dataset(n_per_class=150):
    random.seed(RANDOM_STATE)
    subjects = ["The product", "This phone", "The service", "The movie", "The food",
                "The delivery", "The app", "The staff", "This laptop", "The hotel"]
    pos = ["is absolutely fantastic", "works perfectly and I love it", "exceeded my expectations",
           "is amazing, highly recommended", "was great and very helpful", "is wonderful and worth every rupee",
           "made me really happy", "is excellent quality", "was brilliant, loved it", "is smooth and impressive"]
    neg = ["is terrible and a waste of money", "stopped working, very disappointing", "was awful and rude",
           "is the worst I have ever used", "did not work at all, I hate it", "was slow and extremely frustrating",
           "is poor quality, never buying again", "broke after one day, so bad", "was horrible and overpriced",
           "is not good, very annoying"]
    neu = ["is okay, nothing special", "arrived on Tuesday", "is average and does the job",
           "has a standard design", "was neither good nor bad", "is available in two colours",
           "is what I expected, nothing more", "was fine, just normal", "comes with a one year warranty",
           "is about the same as the others"]
    ep = ["Really happy!", "Will buy again.", "Five stars.", "Totally satisfied.", ""]
    en = ["Very upset.", "Do not buy.", "One star.", "Totally unsatisfied.", ""]
    eu = ["No strong opinion.", "Just sharing.", "Three stars.", "It is fine.", ""]
    rows = []
    for _ in range(n_per_class):
        rows.append((f"{random.choice(subjects)} {random.choice(pos)}. {random.choice(ep)}".strip(), "positive"))
        rows.append((f"{random.choice(subjects)} {random.choice(neg)}. {random.choice(en)}".strip(), "negative"))
        rows.append((f"{random.choice(subjects)} {random.choice(neu)}. {random.choice(eu)}".strip(), "neutral"))
    return pd.DataFrame(rows, columns=["text", "label"]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

if USE_DEMO:
    df = build_demo_dataset()
    print("Using DEMO dataset:", len(df), "rows")
else:
    from google.colab import files
    uploaded = files.upload()
    csv_name = list(uploaded.keys())[0]
    raw = pd.read_csv(csv_name, encoding_errors="ignore")
    print("Columns in your CSV:", list(raw.columns))
    df = raw[[TEXT_COL, LABEL_COL]].dropna()
    df.columns = ["text", "label"]
    print("Loaded", len(df), "rows")

df["label"] = df["label"].astype(str).str.strip().str.lower()
print(df["label"].value_counts())
df.head()

## Step 3: Preprocessing (stop-words + lemmatization)

Negations like *not* and *no* are kept because they flip sentiment ("not good" ≠ "good").

In [ ]:
class TextPreprocessor:
    def __init__(self):
        base = set(stopwords.words("english"))
        keep = {"no", "nor", "not", "don", "don't", "didn", "didn't", "doesn", "doesn't",
                "isn", "isn't", "wasn", "wasn't", "won", "won't", "couldn", "couldn't",
                "shouldn", "shouldn't", "wouldn", "wouldn't", "aren", "aren't", "hasn",
                "hasn't", "haven", "haven't", "very", "too", "against"}
        self.stop = base - keep
        self.lemmatizer = WordNetLemmatizer()
        try:
            nltk.pos_tag(["test"]); self.use_pos = True
        except Exception:
            self.use_pos = False

    @staticmethod
    def _wn_pos(tag):
        return {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

    def clean(self, text):
        text = str(text).lower()
        text = re.sub(r"http\S+|www\.\S+", " ", text)
        text = re.sub(r"@\w+", " ", text)
        text = re.sub(r"#", " ", text)
        text = re.sub(r"n't\b", " not", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def __call__(self, text):
        tokens = [t for t in self.clean(text).split() if len(t) > 1 and t not in self.stop]
        if self.use_pos and tokens:
            tokens = [self.lemmatizer.lemmatize(w, self._wn_pos(t)) for w, t in nltk.pos_tag(tokens)]
        else:
            tokens = [self.lemmatizer.lemmatize(w) for w in tokens]
        return " ".join(tokens)

prep = TextPreprocessor()
df["clean"] = df["text"].apply(prep)
df = df[df["clean"].str.len() > 0].reset_index(drop=True)

print("BEFORE:", df["text"][0])
print("AFTER :", df["clean"][0])

## Step 4: Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, stratify=df["label"], random_state=RANDOM_STATE)
labels = sorted(df["label"].unique())
print("Train:", len(X_train), "| Test:", len(X_test), "| Classes:", labels)

## Step 5: TF-IDF + train and compare three models

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "LinearSVM": LinearSVC(class_weight="balanced"),
    "NaiveBayes": MultinomialNB(),
}

results, fitted = [], {}
for name, clf in models.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95, sublinear_tf=True)),
        ("clf", clf),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
        "f1_weighted": f1_score(y_test, pred, average="weighted"),
    })
    fitted[name] = (pipe, pred)

comparison = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
comparison.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)
comparison

## Step 6: Evaluate the best model (F1-score)

In [ ]:
best_name = comparison.loc[0, "model"]
best_pipe, best_pred = fitted[best_name]
print("Best model:", best_name, "\n")

report_txt = classification_report(y_test, best_pred, labels=labels, digits=4)
report_dict = classification_report(y_test, best_pred, labels=labels, output_dict=True)
print(report_txt)
with open(f"{OUT_DIR}/classification_report.txt", "w") as f:
    f.write(f"Best model: {best_name}\n\n{report_txt}")

# Confusion matrix
cm = confusion_matrix(y_test, best_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=200); plt.show()

# Per-class F1
scores = [report_dict[l]["f1-score"] for l in labels]
plt.figure(figsize=(6, 4))
bars = plt.bar(labels, scores, color="#4C72B0")
plt.ylim(0, 1.05); plt.ylabel("F1-score"); plt.title(f"Per-class F1 - {best_name}")
for b, s in zip(bars, scores):
    plt.text(b.get_x() + b.get_width() / 2, s + 0.01, f"{s:.2f}", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/f1_per_class.png", dpi=200); plt.show()

with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump({"best_model": best_name, "labels": labels, "results": results, "report": report_dict}, f, indent=2)

## Step 7: Try your own sentences

In [ ]:
def predict_sentiment(sentence):
    return best_pipe.predict([prep(sentence)])[0]

tests = ["I absolutely love this, it works great!",
         "This is the worst thing I have ever bought.",
         "It arrived on Monday in a brown box.",
         "The flight was not good and the staff were rude."]
for s in tests:
    print(f"{predict_sentiment(s):9s} <- {s}")

## Step 8: Download all results

This zips the `results/` folder (reports, charts, metrics) so you can add it to GitHub.

In [ ]:
import joblib
joblib.dump({"pipeline": best_pipe, "labels": labels}, f"{OUT_DIR}/best_model.joblib")

with zipfile.ZipFile("task2_results.zip", "w") as z:
    for fn in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, fn))
print("Saved task2_results.zip ✅")

try:
    from google.colab import files
    files.download("task2_results.zip")
except ImportError:
    pass